# 🌍 Domain Generalization & Robustness Analysis

## 🎯 Objective

Evaluate the robustness and generalization capability of the trained model across different real-world domains.

## Why this matters

In production, data distributions change across:

* Countries
* Time periods
* Customer behavior segments

A high-performing model must:
✔ Generalize across domains
✔ Avoid reliance on spurious correlations
✔ Maintain stable performance

---

## 🔍 This notebook evaluates:

1. Geographic generalization (country-level)
2. Temporal generalization (time-based)
3. Behavioral generalization (customer segments)
4. Performance stability across domains


**1. IMPORTS**

In [1]:
# ======================
# IMPORTS
# ======================
import sys, os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np

from sklearn.metrics import f1_score
from lightgbm import LGBMClassifier

from src.pipeline import full_pipeline
from src.data_loader import load_data
from src.config import load_config

import warnings
warnings.filterwarnings("ignore")

**2. LOAD CONFIG**

In [2]:
# ======================
# LOAD CONFIG
# ======================
config = load_config()

RANDOM_STATE = config["training"]["random_state"]

**3. LOAD DATA + PIPELINE**

In [3]:
# ======================
# LOAD DATA
# ======================
train, test, _ = load_data()

train_processed, test_processed = full_pipeline(train, test)

print("Train shape:", train_processed.shape)

Train shape: (68654, 26)


**4. FEATURE SELECTION (LEAKAGE SAFE)**

In [4]:
# ======================
# FEATURE SELECTION
# ======================
DROP_COLS = [
    "ID",
    "target",
    "customer_id",
    "tbl_loan_id",
    "lender_id",
    "disbursement_date",
    "due_date"
]

FEATURES = [col for col in train_processed.columns if col not in DROP_COLS]

X = train_processed[FEATURES]
y = train_processed["target"]

print("Number of features:", len(FEATURES))

Number of features: 19


**5. CLASS IMBALANCE HANDLING**

In [5]:
pos = y.sum()
neg = len(y) - pos

scale_pos_weight = neg / pos

print("Scale_pos_weight:", scale_pos_weight)

Scale_pos_weight: 53.573926868044516


**6. MODEL INITIALIZATION**

In [6]:
model = LGBMClassifier(
    n_estimators=935,
    learning_rate=0.103,
    num_leaves=82,
    max_depth=4,
    min_child_samples=85,
    subsample=0.63,
    colsample_bytree=0.97,
    reg_alpha=0.082,
    reg_lambda=0.029,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight
)

**7. DOMAIN 1 - GEOGRAPHIC GENERALIZATION**

**Leave-One-Country-Out Validation**

In [13]:
print(train_processed["country_id"].value_counts())

country_id
0    68654
Name: count, dtype: int64


In [14]:
# ==============================
# 🌍 COUNTRY GENERALIZATION (SAFE VERSION)
# ==============================

from sklearn.metrics import f1_score
import pandas as pd

print("🌍 Running Country Generalization Check...")

# ------------------------------
# STEP 1: Validate country diversity
# ------------------------------
country_counts = train_processed["country_id"].value_counts()
n_countries = train_processed["country_id"].nunique()

print("\nCountry distribution:")
print(country_counts)

# ------------------------------
# STEP 2: Handle single-country case (CRITICAL)
# ------------------------------
if n_countries < 2:
    print("\n⚠️ ONLY ONE COUNTRY DETECTED")
    print("👉 Geographic generalization is NOT applicable")
    print("👉 This dataset does not support cross-country validation")
    
    country_df = pd.DataFrame({
        "status": ["SKIPPED"],
        "reason": ["Single-country dataset"],
        "recommendation": ["Use temporal + behavioral validation instead"]
    })

    display(country_df)

else:
    # ------------------------------
    # STEP 3: Filter valid countries
    # ------------------------------
    MIN_SAMPLES = 50

    valid_countries = country_counts[country_counts > MIN_SAMPLES].index.tolist()

    print(f"\nValid countries (>{MIN_SAMPLES} samples): {len(valid_countries)}")

    # ------------------------------
    # STEP 4: Reset index (VERY IMPORTANT)
    # ------------------------------
    X_safe = X.reset_index(drop=True)
    y_safe = y.reset_index(drop=True)
    df_safe = train_processed.reset_index(drop=True)

    # ------------------------------
    # STEP 5: Run Leave-One-Country-Out CV
    # ------------------------------
    country_results = []

    for country in valid_countries:
        print(f"\n🔍 Evaluating country: {country}")

        train_idx = df_safe["country_id"] != country
        val_idx = df_safe["country_id"] == country

        X_train, X_val = X_safe[train_idx], X_safe[val_idx]
        y_train, y_val = y_safe[train_idx], y_safe[val_idx]

        # Safety check
        if X_train.shape[0] == 0 or X_val.shape[0] == 0:
            print(f"⚠️ Skipping country {country} (empty split)")
            continue

        model.fit(X_train, y_train)
        preds = model.predict(X_val)

        score = f1_score(y_val, preds)

        country_results.append({
            "country": country,
            "f1_score": score,
            "n_samples": len(y_val)
        })

    # ------------------------------
    # STEP 6: Handle results safely
    # ------------------------------
    country_df = pd.DataFrame(country_results)

    if not country_df.empty:
        country_df = country_df.sort_values("f1_score")
        print("\n📊 Country Generalization Results:")
        display(country_df)
    else:
        print("\n⚠️ No valid country splits were produced")

🌍 Running Country Generalization Check...

Country distribution:
country_id
0    68654
Name: count, dtype: int64

⚠️ ONLY ONE COUNTRY DETECTED
👉 Geographic generalization is NOT applicable
👉 This dataset does not support cross-country validation


,status,reason,recommendation
0,SKIPPED,Single-country dataset,Use temporal + behavioral validation instead


**8. DOMAIN 2 - TEMPORAL GENERALIZATION**

In [16]:
# ======================
# TIME-BASED GENERALIZATION
# ======================
df_sorted = train_processed.sort_values("disbursement_date")

split_point = int(len(df_sorted) * 0.7)

train_part = df_sorted.iloc[:split_point]
val_part = df_sorted.iloc[split_point:]

X_train = train_part[FEATURES]
y_train = train_part["target"]

X_val = val_part[FEATURES]
y_val = val_part["target"]

model.fit(X_train, y_train)

preds = model.predict(X_val)
time_f1 = f1_score(y_val, preds)

print("Time-based F1:", time_f1)

[LightGBM] [Info] Number of positive: 666, number of negative: 47391
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003308 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2094
[LightGBM] [Info] Number of data points in the train set: 48057, number of used features: 17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.013859 -> initscore=-4.264898
[LightGBM] [Info] Start training from score -4.264898
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

**9. DOMAIN 3 - BEHAVIORAL GENERALIZATION**

In [19]:
# ======================
# 👥 BEHAVIORAL GENERALIZATION (FIXED + PRODUCTION)
# ======================

from sklearn.metrics import f1_score
import pandas as pd

print("👥 Running Behavioral Generalization...")

# ------------------------------
# STEP 1: Create aligned dataset
# ------------------------------
df_safe = train_processed.copy()

df_safe["segment"] = pd.cut(
    df_safe["loan_count"],
    bins=[-1, 2, 5, 100],
    labels=["low", "medium", "high"]
)

# ------------------------------
# 🚨 CRITICAL FIX: ALIGN DATA FIRST
# ------------------------------
# Keep only rows with valid segment
valid_idx = df_safe["segment"].notna()

df_safe = df_safe.loc[valid_idx].reset_index(drop=True)
X_safe = X.loc[valid_idx].reset_index(drop=True)
y_safe = y.loc[valid_idx].reset_index(drop=True)

print("\nSegment distribution:")
print(df_safe["segment"].value_counts())

# ------------------------------
# STEP 2: Run validation
# ------------------------------
segment_results = []

for seg in df_safe["segment"].unique():

    print(f"\n🔍 Evaluating segment: {seg}")

    val_idx = df_safe["segment"] == seg
    train_idx = ~val_idx

    X_train, X_val = X_safe.loc[train_idx], X_safe.loc[val_idx]
    y_train, y_val = y_safe.loc[train_idx], y_safe.loc[val_idx]

    # ------------------------------
    # 🚨 SAFETY CHECKS (CRITICAL)
    # ------------------------------
    if X_val.shape[0] == 0:
        print(f"⚠️ Skipping {seg} → empty validation set")
        continue

    if X_train.shape[0] == 0:
        print(f"⚠️ Skipping {seg} → empty training set")
        continue

    if len(y_val.unique()) < 2:
        print(f"⚠️ Skipping {seg} → only one class")
        continue

    # ------------------------------
    # Train + evaluate
    # ------------------------------
    model.fit(X_train, y_train)

    preds = model.predict(X_val)
    score = f1_score(y_val, preds)

    segment_results.append({
        "segment": str(seg),
        "f1_score": score,
        "n_samples": len(y_val)
    })

# ------------------------------
# STEP 3: Safe output
# ------------------------------
segment_df = pd.DataFrame(segment_results)

if not segment_df.empty:
    segment_df = segment_df.sort_values("f1_score")
    print("\n📊 Behavioral Generalization Results:")
    display(segment_df)
else:
    print("\n⚠️ No valid segment splits were produced")

👥 Running Behavioral Generalization...

Segment distribution:
segment
high      40362
low       16484
medium    11483
Name: count, dtype: int64

🔍 Evaluating segment: low
[LightGBM] [Info] Number of positive: 541, number of negative: 51304
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002916 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2087
[LightGBM] [Info] Number of data points in the train set: 51845, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.010435 -> initscore=-4.552105
[LightGBM] [Info] Start training from score -4.552105
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posi

,segment,f1_score,n_samples
0,low,0.753580,16484
2,high,0.791209,40362
1,medium,0.806100,11483


**DOMAIN STABILITY SUMMARY**

In [21]:
summary_rows = []

# ------------------------------
# COUNTRY (only if valid)
# ------------------------------
if "f1_score" in country_df.columns:
    summary_rows.append({
        "Domain": "Country",
        "Mean F1": country_df["f1_score"].mean(),
        "Std/Variation": country_df["f1_score"].std()
    })
else:
    summary_rows.append({
        "Domain": "Country",
        "Mean F1": None,
        "Std/Variation": None
    })

# ------------------------------
# TIME (always valid)
# ------------------------------
summary_rows.append({
    "Domain": "Time",
    "Mean F1": time_f1,
    "Std/Variation": None
})

# ------------------------------
# BEHAVIOR (only if valid)
# ------------------------------
if "f1_score" in segment_df.columns:
    summary_rows.append({
        "Domain": "Behavior",
        "Mean F1": segment_df["f1_score"].mean(),
        "Std/Variation": segment_df["f1_score"].std()
    })
else:
    summary_rows.append({
        "Domain": "Behavior",
        "Mean F1": None,
        "Std/Variation": None
    })

# ------------------------------
# FINAL TABLE
# ------------------------------
summary = pd.DataFrame(summary_rows)

print("\n📊 Domain Stability Summary:")
display(summary)


📊 Domain Stability Summary:


,Domain,Mean F1,Std/Variation
0,Country,NaN,NaN
1,Time,0.642654,NaN
2,Behavior,0.783630,0.027068


**BIAS & SPURIOUS CORRELATION CHECK**

In [22]:
# Target distribution per country
country_target = train_processed.groupby("country_id")["target"].mean().reset_index()
display(country_target.sort_values("target", ascending=False))

,country_id,target
0,0,0.018324


In [23]:
# Target distribution over time
time_target = train_processed.groupby("year")["target"].mean().reset_index()
display(time_target)

,year,target
0,2021,0.486486
1,2022,0.013258
2,2023,0.113617
3,2024,0.085012


## 🔍 Domain Generalization Analysis — Summary

This phase evaluated the model’s robustness across multiple real-world dimensions: geographic, temporal, and behavioral domains.

### 🌍 Country Generalization
The dataset contains only a single country, making geographic generalization infeasible. The pipeline correctly detects this limitation and avoids performing invalid cross-country validation. This ensures that model evaluation remains realistic and not artificially inflated.

### 👥 Behavioral Generalization (Customer Segments)
The model demonstrates strong and stable performance across customer segments:

- Low activity customers: F1 ≈ 0.75  
- Medium activity customers: F1 ≈ 0.81  
- High activity customers: F1 ≈ 0.79  

The low standard deviation (~0.027) indicates minimal performance variance, suggesting that the model generalizes well across different customer behavior profiles. Slightly lower performance in the “low” segment is expected due to limited historical information.

### ⏳ Temporal Generalization (Critical Finding)
Time-based validation reveals a significantly lower F1 score (~0.64) compared to earlier cross-validation results. This indicates that the model struggles to generalize to future data, highlighting a key limitation in real-world deployment scenarios.

### 📉 Temporal Drift Analysis
A substantial shift in target distribution over time was observed:

- 2021: ~48.6% default rate  
- 2022–2024: significantly lower and fluctuating  

This confirms strong temporal drift in the dataset, meaning the underlying data distribution changes over time. As a result, patterns learned from earlier periods do not fully transfer to later periods.

### 🧠 Key Conclusion
The model is:
- **Behaviorally robust** (generalizes well across customer types)
- **Temporally fragile** (sensitive to distribution shifts over time)

This highlights the importance of incorporating time-aware validation and potentially adapting the model to handle temporal drift.

### 🚀 Implications
To improve real-world performance, future work should focus on:
- Time-aware feature engineering  
- Incorporating macroeconomic or external signals  
- Dynamic or rolling retraining strategies  

Overall, this analysis provides a realistic and production-oriented assessment of model reliability.